In [1]:
"""
NeuralProt — Training Pipeline
=============================
Loads per-group processed data (features.npy, labels.npy, go_terms.json)
and trains a separate ProteinMLP model for each biological GO term group.

Split: 80% train / 10% validation / 10% test (held out, never used here).
Each group's test indices are saved to test_idx.json for final evaluation.

Saves per group:
  {group}_best.pt    — best model weights by validation F1
  {group}_resume.pt  — full checkpoint for resuming interrupted training
  {group}_log.json   — per-epoch metrics
"""

import os
import json
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
 
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
 
print(f"PyTorch version : {torch.__version__}")
print(f"Device available: {'CUDA' if torch.cuda.is_available() else 'CPU'}")
 
 
PROCESSED_DIR = "C:/Users/USER/Documents/cod3astro/ML_AI/NeuralProt/data/processed/processed_data"
OUTPUT_DIR    = "C:/Users/USER/Documents/cod3astro/ML_AI/NeuralProt/model/models"
 
BATCH_SIZE   = 32
EPOCHS       = 100
LR           = 0.001
WEIGHT_DECAY = 1e-4
POS_WEIGHT_CLIP = 10000  # maximum pos_weight value allowed
 
# To train only specific groups, edit this list.
# To train all 22, leave the list as it is.
TRAIN_GROUPS = [
    "reproductive_process",
    "interspecies_interaction",
    "immune_system_process",
    "molecular_transducer",
    "mf_regulator_activity",
    "homeostatic_process",
    "atp_dependent_activity",
    "lipid_transport",
    "nuclear_transport",
    "vesicle_mediated_transport",
    "protein_transport",
    "ion_transport",
    "oxidoreductase_activity",
    "transferase_activity",
    "hydrolase_activity",
    "lyase_activity",
    "protein_binding",
    "dna_binding",
    "rna_binding",
    "lipid_binding",
    "metal_ion_binding",
    "small_molecule_binding",
]
 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
 
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✓ Config set")
print(f"  Device     : {DEVICE}")
print(f"  Output dir : {OUTPUT_DIR}/")

PyTorch version : 2.9.1+cpu
Device available: CPU
✓ Config set
  Device     : cpu
  Output dir : C:/Users/USER/Documents/cod3astro/ML_AI/NeuralProt/model/models/


In [2]:
# Quick check: verify all requested groups exist on disk
print("Checking processed data availability...")
missing_groups = []
for gname in TRAIN_GROUPS:
    group_dir = os.path.join(PROCESSED_DIR, gname)
    if not os.path.isdir(group_dir):
        missing_groups.append(gname)
        print(f"  ✗ {gname} — NOT FOUND in {PROCESSED_DIR}/")
    else:
        print(f"  ✓ {gname}")
 
if missing_groups:
    print(f"\n⚠  Run data_processor.py first for: {missing_groups}")
else:
    print(f"\n✓ All groups ready to train")

Checking processed data availability...
  ✓ reproductive_process
  ✓ interspecies_interaction
  ✓ immune_system_process
  ✓ molecular_transducer
  ✓ mf_regulator_activity
  ✓ homeostatic_process
  ✓ atp_dependent_activity
  ✓ lipid_transport
  ✓ nuclear_transport
  ✓ vesicle_mediated_transport
  ✓ protein_transport
  ✓ ion_transport
  ✓ oxidoreductase_activity
  ✓ transferase_activity
  ✓ hydrolase_activity
  ✓ lyase_activity
  ✓ protein_binding
  ✓ dna_binding
  ✓ rna_binding
  ✓ lipid_binding
  ✓ metal_ion_binding
  ✓ small_molecule_binding

✓ All groups ready to train


In [3]:
class ProteinDataset(Dataset):
    """
    Dataset that loads precomputed 428-dimensional protein feature vectors.
    No tokenization needed — features are already numerical.
    """
 
    def __init__(self, indices, feature_matrix, label_matrix):
        self.indices        = indices
        self.feature_matrix = feature_matrix
        self.label_matrix   = label_matrix
 
    def __len__(self):
        return len(self.indices)
 
    def __getitem__(self, idx):
        i       = self.indices[idx]
        features = torch.tensor(self.feature_matrix[i], dtype=torch.float32)
        label    = torch.tensor(self.label_matrix[i],   dtype=torch.float32)
        return features, label
 
 
def load_group_data(group_name, processed_dir):
    """
    Load all processed data for a group including feature matrix.
    """
    group_dir = os.path.join(processed_dir, group_name)
 
    required = ["accessions.json", "labels.npy", "go_terms.json",
                "features.npy", "metadata.json"]
    for fname in required:
        fpath = os.path.join(group_dir, fname)
        if not os.path.exists(fpath):
            raise FileNotFoundError(
                f"Missing: {fpath}\n"
                f"Have you run data_processor.py (including Cell 8b) "
                f"for group '{group_name}'?"
            )
 
    with open(os.path.join(group_dir, "accessions.json")) as f:
        accessions = json.load(f)
 
    label_matrix   = np.load(os.path.join(group_dir, "labels.npy"))
    feature_matrix = np.load(os.path.join(group_dir, "features.npy"))
 
    with open(os.path.join(group_dir, "go_terms.json")) as f:
        go_term_list = json.load(f)
 
    with open(os.path.join(group_dir, "metadata.json")) as f:
        metadata = json.load(f)
 
    return accessions, feature_matrix, label_matrix, go_term_list, metadata
 
 
def build_dataloaders(feature_matrix, label_matrix, batch_size, group_dir):
    """
    Split into train/val/test by index and build DataLoaders.
    random_state=42 must never change — required for checkpoint resume.

    Split ratio: 80% train / 10% val / 10% test
    test_split is held out entirely — not used for training or threshold tuning.
    Test indices are saved to group_dir/test_idx.json for evaluation later.
    """
    n       = len(label_matrix)
    indices = list(range(n))

    # First carve out the test set (10%)
    train_val_idx, test_idx = train_test_split(
        indices, test_size=0.1, random_state=42
    )

    # Then split the remaining 90% into 80% train / 10% val
    # 0.111 * 90% ≈ 10% of total
    train_idx, val_idx = train_test_split(
        train_val_idx, test_size=0.111, random_state=42
    )

    # Save test indices to disk so evaluate_with_fmax can use them later
    # This file must never be used during training or threshold tuning
    test_idx_path = os.path.join(group_dir, "test_idx.json")
    if not os.path.exists(test_idx_path):
        with open(test_idx_path, "w") as f:
            json.dump(test_idx, f)
        print(f"  Test indices saved → {test_idx_path}  ({len(test_idx)} proteins held out)")
    else:
        print(f"  Test indices already exist — not overwritten (resume safety)")

    train_ds = ProteinDataset(train_idx, feature_matrix, label_matrix)
    val_ds   = ProteinDataset(val_idx,   feature_matrix, label_matrix)

    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=True,  num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size,
                              shuffle=False, num_workers=0)

    return train_loader, val_loader, len(train_ds), len(val_ds)
 
 
def compute_pos_weights(label_matrix, clip_max=10000.0):
    n          = label_matrix.shape[0]
    pos_counts = label_matrix.sum(axis=0) + 1e-6
    neg_counts = n - pos_counts
    weights    = neg_counts / pos_counts
    weights    = np.clip(weights, 1.0, clip_max)
    return torch.tensor(weights, dtype=torch.float32)
 
 
print("✓ Feature-based Dataset class defined")

✓ Feature-based Dataset class defined


In [4]:
class ProteinMLP(nn.Module):
    """
    Feedforward MLP for multi-label GO term prediction.
 
    Input  : 428 precomputed protein features (AAC + DPC + physicochemical)
    Output : num_classes GO term predictions (raw logits)
 
    Architecture:
      Linear 428 → 1024, BatchNorm, ReLU, Dropout(0.3)
      Linear 1024 → 512,  BatchNorm, ReLU, Dropout(0.3)
      Linear 512  → 256,  BatchNorm, ReLU, Dropout(0.2)
      Linear 256  → num_classes
 
    Why MLP over CNN here:
      - Features are already meaningful numerical values, not raw sequence
      - MLP trains 10–50x faster than CNN on CPU
      - BatchNorm stabilises training with sparse multi-label targets
      - Dropout prevents overfitting on smaller groups
    """
 
    def __init__(self, input_dim=428, num_classes=128, dropout=0.3):
        super().__init__()
 
        self.network = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(dropout),
 
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(dropout),
 
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),
 
            nn.Linear(256, num_classes),
        )
 
    def forward(self, x):
        return self.network(x)
 
 
print("✓ ProteinMLP defined")
 

# Show parameter count for smallest and largest actual groups
for n_cls, label in [(13, "nuclear_transport (smallest group)"),
                     (159, "oxidoreductase_activity (largest group)")]:
    m = ProteinMLP(num_classes=n_cls)
    n = sum(p.numel() for p in m.parameters())
    print(f"  {label}: {n:,} parameters")

✓ ProteinMLP defined
  nuclear_transport (smallest group): 1,102,349 parameters
  oxidoreductase_activity (largest group): 1,139,871 parameters


In [5]:
def evaluate(model, loader, criterion, device, threshold=0.5):
    """
    Run one full validation pass.
    Returns: avg_loss, macro_f1, macro_precision, macro_recall
    """
    model.eval()
    total_loss  = 0.0
    all_preds   = []
    all_targets = []
 
    with torch.no_grad():
        for seqs, labels in loader:
            seqs   = seqs.to(device)
            labels = labels.to(device)
 
            logits = model(seqs)
            loss   = criterion(logits, labels)
            total_loss += loss.item()
 
            probs = torch.sigmoid(logits)
            preds = (probs >= threshold).float()
 
            all_preds.append(preds.cpu().numpy())
            all_targets.append(labels.cpu().numpy())
 
    all_preds   = np.vstack(all_preds)
    all_targets = np.vstack(all_targets)
 
    f1        = f1_score(all_targets,        all_preds,
                         average="macro", zero_division=0)
    precision = precision_score(all_targets, all_preds,
                                average="macro", zero_division=0)
    recall    = recall_score(all_targets,    all_preds,
                             average="macro", zero_division=0)
    avg_loss  = total_loss / len(loader)
 
    return avg_loss, f1, precision, recall
 
 
def train_group(group_name, train_loader, val_loader, num_classes,
                pos_weights, device, output_dir, epochs, lr, weight_decay):
    """
    Full training loop for one GO group with checkpoint resume support.
 
    Two checkpoint files are maintained:
      {group}_best.pt   — saved whenever val F1 improves (used for inference)
      {group}_resume.pt — saved after EVERY epoch (used to resume after shutdown)
 
    On startup this function checks for a resume checkpoint. If found, it
    restores model weights, optimizer state, scheduler state, best F1 so far,
    and the training log, then continues from the next epoch.
 
    If training completed fully in a previous session, it returns immediately
    without re-training.
    """
    print(f"\n{'='*62}")
    print(f"  Training: {group_name}  ({num_classes} GO terms)")
    print(f"{'='*62}")
 
    best_path   = os.path.join(output_dir, f"{group_name}_best.pt")
    resume_path = os.path.join(output_dir, f"{group_name}_resume.pt")
    log_path    = os.path.join(output_dir, f"{group_name}_log.json")
 
    # ── Initialise model, optimizer, scheduler 
    model     = ProteinMLP(num_classes=num_classes).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights.to(device))
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=2, factor=0.5
    )
 
    start_epoch = 1
    best_val_f1 = -1.0
    log         = []
 
    # ── Check for existing resume checkpoint 
    if os.path.exists(resume_path):
        print(f"  ✓ Resume checkpoint found — loading...")
        checkpoint = torch.load(resume_path, map_location=device, weights_only=False)
 
        # Check if training already finished in a previous session
        if checkpoint.get("training_complete", False):
            print(f"  ✓ Training already completed in a previous session.")
            print(f"    Best Val F1 : {checkpoint['best_val_f1']:.4f}")
            print(f"    Skipping    : {group_name}")
            return checkpoint["best_val_f1"]
 
        model.load_state_dict(checkpoint["model_state"])
        optimizer.load_state_dict(checkpoint["optimizer_state"])
        scheduler.load_state_dict(checkpoint["scheduler_state"])
        start_epoch = checkpoint["epoch"] + 1      # resume from NEXT epoch
        best_val_f1 = checkpoint["best_val_f1"]
        log         = checkpoint.get("log", [])
 
        print(f"  Resuming from epoch {start_epoch}/{epochs}")
        print(f"  Best val F1 so far : {best_val_f1:.4f}")
 
        if start_epoch > epochs:
            print(f"  ✓ All {epochs} epochs already completed.")
            return best_val_f1
    else:
        print(f"  No checkpoint found — starting from epoch 1")
 
    # ── Training loop 
    for epoch in range(start_epoch, epochs + 1):
        model.train()
        epoch_loss = 0.0
        t_start    = time.time()
 
        for seqs, labels in train_loader:
            seqs   = seqs.to(device)
            labels = labels.to(device)
 
            optimizer.zero_grad()
            logits = model(seqs)
            loss   = criterion(logits, labels)
            loss.backward()
            optimizer.step()
 
            epoch_loss += loss.item()
 
        avg_train_loss = epoch_loss / len(train_loader)
        val_loss, val_f1, val_p, val_r = evaluate(
            model, val_loader, criterion, device
        )
        scheduler.step(val_loss)
        elapsed = time.time() - t_start
 
        # ── Save best model if val F1 improved 
        improved = ""
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save({
                "epoch":       epoch,
                "model_state": model.state_dict(),
                "val_f1":      val_f1,
                "val_loss":    val_loss,
                "num_classes": num_classes,
                "group_name":  group_name,
            }, best_path)
            improved = "  ← best"
 
        print(
            f"  Epoch {epoch:>2}/{epochs} | "
            f"Train: {avg_train_loss:.4f} | "
            f"Val: {val_loss:.4f} | "
            f"F1: {val_f1:.4f} | "
            f"P: {val_p:.4f} | "
            f"R: {val_r:.4f} | "
            f"{elapsed:.0f}s{improved}"
        )
 
        # ── Append to log 
        log.append({
            "epoch":      epoch,
            "train_loss": round(avg_train_loss, 6),
            "val_loss":   round(val_loss, 6),
            "val_f1":     round(val_f1, 6),
            "precision":  round(val_p, 6),
            "recall":     round(val_r, 6),
        })
 
        # ── Save log to disk after every epoch 
        # Written every epoch so partial logs survive a shutdown.
        with open(log_path, "w") as f:
            json.dump(log, f, indent=2)
 
        # ── Save resume checkpoint after every epoch 

        training_complete = (epoch == epochs)
        torch.save({
            "epoch":             epoch,
            "model_state":       model.state_dict(),
            "optimizer_state":   optimizer.state_dict(),
            "scheduler_state":   scheduler.state_dict(),
            "best_val_f1":       best_val_f1,
            "num_classes":       num_classes,
            "group_name":        group_name,
            "log":               log,
            "training_complete": training_complete,
        }, resume_path)
 
    print(f"\n  Best Val F1 : {best_val_f1:.4f}")
    print(f"  Best model  : {best_path}")
    print(f"  Resume ckpt : {resume_path}")
    print(f"  Log         : {log_path}")
 
    return best_val_f1
 
 
print("✓ Training functions defined (with checkpoint resume)")

✓ Training functions defined (with checkpoint resume)


In [6]:
summary = {}
 
for group_name in TRAIN_GROUPS:
    try:
        # ── Load processed data 
        accessions, feature_matrix, label_matrix, go_term_list, metadata = \
            load_group_data(group_name, PROCESSED_DIR)
 
        print(f"\nLoaded: {group_name}")
        print(f"  Proteins     : {len(accessions):,}")
 
        # ── Class weights 
        pos_weights = compute_pos_weights(label_matrix, clip_max=POS_WEIGHT_CLIP)
        print(f"  pos_weight — min: {pos_weights.min():.1f}  "
              f"max: {pos_weights.max():.1f}  "
              f"mean: {pos_weights.mean():.1f}")
 
        # ── DataLoaders 
        group_dir = os.path.join(PROCESSED_DIR, group_name)

        # random_state=42 ensures the same train/val split every run
        train_loader, val_loader, n_train, n_val = build_dataloaders(
                feature_matrix, label_matrix, BATCH_SIZE, group_dir,
        )
        n_test = len(label_matrix) - n_train - n_val
        print(f"  Train: {n_train:,}  |  Val: {n_val:,}  |  Test: {n_test:,} (held out)")
 
        # ── Train (or resume) 
        best_f1 = train_group(
            group_name   = group_name,
            train_loader = train_loader,
            val_loader   = val_loader,
            num_classes  = len(go_term_list),
            pos_weights  = pos_weights,
            device       = DEVICE,
            output_dir   = OUTPUT_DIR,
            epochs       = EPOCHS,
            lr           = LR,
            weight_decay = WEIGHT_DECAY,
        )
 
        summary[group_name] = {
            "n_proteins":  len(accessions),
            "n_go_terms":  len(go_term_list),
            "best_val_f1": round(best_f1, 4),
            "status":      "completed",
        }
 
    except Exception as e:
        print(f"\n✗ {group_name} failed: {e}")
        summary[group_name] = {"status": "failed", "error": str(e)}


Loaded: reproductive_process
  Proteins     : 6,273
  pos_weight — min: 1.0  max: 124.5  mean: 62.5
  Test indices already exist — not overwritten (resume safety)
  Train: 5,018  |  Val: 627  |  Test: 628 (held out)

  Training: reproductive_process  (72 GO terms)
  ✓ Resume checkpoint found — loading...
  ✓ Training already completed in a previous session.
    Best Val F1 : 0.2154
    Skipping    : reproductive_process

Loaded: interspecies_interaction
  Proteins     : 8,924
  pos_weight — min: 1.0  max: 177.5  mean: 71.9
  Test indices already exist — not overwritten (resume safety)
  Train: 7,139  |  Val: 892  |  Test: 893 (held out)

  Training: interspecies_interaction  (111 GO terms)
  ✓ Resume checkpoint found — loading...
  ✓ Training already completed in a previous session.
    Best Val F1 : 0.3952
    Skipping    : interspecies_interaction

Loaded: immune_system_process
  Proteins     : 5,047
  pos_weight — min: 1.0  max: 98.0  mean: 42.2
  Test indices already exist — not o